In [8]:
import pandas as pd

# Columns known at the time the loan was approved, plus the outcome
keep = [
    "id", "loan_amnt", "term", "int_rate", "installment", "grade", "sub_grade",
    "emp_length", "home_ownership", "annual_inc", "verification_status",
    "issue_d", "loan_status", "purpose", "addr_state", "dti",
    "delinq_2yrs", "fico_range_low", "fico_range_high", "open_acc",
    "pub_rec", "revol_bal", "revol_util", "total_acc", "application_type",
    # used only for loss analysis, not for predicting
    "total_pymnt", "recoveries",
]

chunks = pd.read_csv(
    "accepted_2007_to_2018Q4.csv",
    usecols=keep,
    chunksize=200_000,
    low_memory=False,
)

done = ["Fully Paid", "Charged Off"]  # loans with a final outcome

df = pd.concat(c[c["loan_status"].isin(done)] for c in chunks)

df["is_default"] = (df["loan_status"] == "Charged Off").astype(int)
df["term"] = df["term"].str.strip()

print(df.shape)
print(df["is_default"].mean())
print(df.isna().sum().sort_values(ascending=False).head(10))

df.to_csv("loans_clean.csv", index=False)

(1345310, 28)
0.19962610848057324
emp_length        78511
revol_util          857
dti                 374
id                    0
int_rate              0
installment           0
grade                 0
sub_grade             0
home_ownership        0
annual_inc            0
dtype: int64


In [10]:
con.execute("""
    CREATE OR REPLACE TABLE loans AS
    SELECT
        * EXCLUDE (issue_d),
        strptime(CAST(issue_d AS VARCHAR), '%b-%Y')::DATE AS issue_date,
        CAST(replace(CAST(int_rate AS VARCHAR), '%', '') AS DOUBLE) AS int_rate_pct
    FROM read_csv_auto('loans_clean.csv')
""")

print(con.execute("SELECT COUNT(*) FROM loans").fetchone())
print(con.execute("""
    SELECT loan_status, COUNT(*) AS n, ROUND(AVG(is_default) * 100, 1) AS default_pct
    FROM loans GROUP BY loan_status
""").df())

(1345310,)
   loan_status        n  default_pct
0   Fully Paid  1076751          0.0
1  Charged Off   268559        100.0


In [16]:
def q(sql):
    return con.execute(sql).df()

In [18]:
q("""
SELECT
    grade,
    COUNT(*) AS loans,
    ROUND(AVG(is_default) * 100, 1) AS default_pct,
    ROUND(AVG(int_rate_pct), 1) AS avg_rate_pct,
    ROUND((SUM(total_pymnt) - SUM(loan_amnt)) / SUM(loan_amnt) * 100, 1) AS net_return_pct
FROM loans
GROUP BY grade
ORDER BY grade
""")

,grade,loans,default_pct,avg_rate_pct,net_return_pct
0,A,235090,6.0,7.1,5.3
1,B,392741,13.4,10.7,5.2
2,C,381686,22.4,14.0,2.7
3,D,200953,30.4,17.7,0.5
4,E,93650,38.5,21.1,-1.4
5,F,32058,45.2,24.9,-4.0
6,G,9132,49.9,27.7,-8.8


In [19]:
q("""
SELECT
    term,
    grade,
    COUNT(*) AS loans,
    ROUND(AVG(is_default) * 100, 1) AS default_pct,
    ROUND((SUM(total_pymnt) - SUM(loan_amnt)) / SUM(loan_amnt) * 100, 1) AS net_return_pct
FROM loans
GROUP BY term, grade
ORDER BY term, grade
""")

,term,grade,loans,default_pct,net_return_pct
0,36 months,A,228803,5.9,5.4
1,36 months,B,344933,12.7,5.6
2,36 months,C,276114,20.5,3.5
3,36 months,D,126058,26.9,1.2
4,36 months,E,36034,32.7,-1.2
5,36 months,F,7282,38.1,-3.7
6,36 months,G,1519,47.9,-12.9
7,60 months,A,6287,10.6,3.4
8,60 months,B,47808,18.0,3.3
9,60 months,C,105572,27.5,1.6


In [20]:
q("""
SELECT
    EXTRACT(year FROM issue_date) AS issue_year,
    COUNT(*) AS loans,
    ROUND(AVG(is_default) * 100, 1) AS default_pct,
    ROUND((SUM(total_pymnt) - SUM(loan_amnt)) / SUM(loan_amnt) * 100, 1) AS net_return_pct
FROM loans
GROUP BY issue_year
ORDER BY issue_year
""")

,issue_year,loans,default_pct,net_return_pct
0,2007,251,17.9,-0.3
1,2008,1562,15.8,-2.3
2,2009,4716,12.6,8.4
3,2010,11536,12.9,7.0
4,2011,21721,15.2,10.6
5,2012,53367,16.2,12.3
6,2013,134804,15.6,14.2
7,2014,223102,18.4,9.4
8,2015,375545,20.2,3.7
9,2016,293095,23.3,-1.7


In [21]:
q("""
SELECT
    purpose,
    COUNT(*) AS loans,
    ROUND(AVG(is_default) * 100, 1) AS default_pct,
    ROUND((SUM(total_pymnt) - SUM(loan_amnt)) / SUM(loan_amnt) * 100, 1) AS net_return_pct
FROM loans
GROUP BY purpose
HAVING COUNT(*) > 5000
ORDER BY default_pct DESC
""")

,purpose,loans,default_pct,net_return_pct
0,small_business,15416,29.7,-3.0
1,moving,9480,23.4,-1.7
2,house,7253,21.9,-1.3
3,medical,15554,21.8,-1.4
4,debt_consolidation,780321,21.1,2.8
5,other,77875,21.0,0.1
6,vacation,9065,19.2,-1.8
7,major_purchase,29425,18.6,-2.0
8,home_improvement,87504,17.7,2.2
9,credit_card,295279,16.9,4.4


In [22]:
q("""
SELECT
    EXTRACT(year FROM issue_date) AS issue_year,
    COUNT(*) AS loans,
    ROUND(AVG(is_default) * 100, 1) AS default_pct,
    ROUND(AVG(int_rate_pct), 1) AS avg_rate_pct,
    ROUND((SUM(total_pymnt) - SUM(loan_amnt)) / SUM(loan_amnt) * 100, 1) AS net_return_pct
FROM loans
WHERE term = '36 months' AND issue_date < DATE '2016-01-01'
GROUP BY issue_year
ORDER BY issue_year
""")

,issue_year,loans,default_pct,avg_rate_pct,net_return_pct
0,2007,251,17.9,10.3,-0.3
1,2008,1562,15.8,11.2,-2.3
2,2009,4716,12.6,12.2,8.4
3,2010,8466,9.9,11.1,7.1
4,2011,14101,10.6,10.6,9.0
5,2012,43470,13.6,12.6,10.2
6,2013,100422,12.3,13.4,11.1
7,2014,162570,13.7,12.5,9.0
8,2015,283026,14.9,11.3,6.6


In [23]:
q("""
SELECT
    purpose,
    COUNT(*) AS loans,
    ROUND(AVG(CASE WHEN grade IN ('E','F','G') THEN 1 ELSE 0 END) * 100, 1) AS pct_grade_E_to_G,
    ROUND(AVG(is_default) * 100, 1) AS default_pct
FROM loans
GROUP BY purpose
HAVING COUNT(*) > 5000
ORDER BY pct_grade_E_to_G DESC
""")

,purpose,loans,pct_grade_E_to_G,default_pct
0,small_business,15416,25.5,29.7
1,house,7253,23.4,21.9
2,moving,9480,17.4,23.4
3,other,77875,14.9,21.0
4,medical,15554,12.2,21.8
5,debt_consolidation,780321,11.1,21.1
6,home_improvement,87504,9.3,17.7
7,vacation,9065,9.2,19.2
8,major_purchase,29425,9.0,18.6
9,car,14585,6.3,14.7


In [24]:
q("""
SELECT
    grade,
    COUNT(*) FILTER (WHERE purpose = 'small_business') AS small_biz_loans,
    ROUND(AVG(CASE WHEN purpose = 'small_business' THEN is_default END) * 100, 1) AS small_biz_default_pct,
    ROUND(AVG(CASE WHEN purpose <> 'small_business' THEN is_default END) * 100, 1) AS other_default_pct
FROM loans
GROUP BY grade
ORDER BY grade
""")

,grade,small_biz_loans,small_biz_default_pct,other_default_pct
0,A,1443,12.3,6.0
1,B,2544,20.8,13.3
2,C,3920,26.4,22.4
3,D,3576,32.8,30.3
4,E,2337,39.9,38.4
5,F,1109,43.7,45.3
6,G,487,50.7,49.9


In [25]:
q("""
SELECT
    CASE
        WHEN dti < 10 THEN '1: under 10'
        WHEN dti < 20 THEN '2: 10 to 20'
        WHEN dti < 30 THEN '3: 20 to 30'
        WHEN dti < 40 THEN '4: 30 to 40'
        ELSE '5: 40 and over'
    END AS dti_band,
    COUNT(*) AS loans,
    ROUND(AVG(is_default) * 100, 1) AS default_pct,
    ROUND((SUM(total_pymnt) - SUM(loan_amnt)) / SUM(loan_amnt) * 100, 1) AS net_return_pct
FROM loans
WHERE dti >= 0
GROUP BY dti_band
ORDER BY dti_band
""")

,dti_band,loans,default_pct,net_return_pct
0,1: under 10,245712,14.9,3.8
1,2: 10 to 20,561904,17.8,4.1
2,3: 20 to 30,408641,23.0,1.9
3,4: 30 to 40,121907,29.1,-1.6
4,5: 40 and over,6770,30.5,-10.0


In [26]:
q("""
SELECT
    CASE
        WHEN annual_inc < 40000 THEN '1: under 40k'
        WHEN annual_inc < 60000 THEN '2: 40k to 60k'
        WHEN annual_inc < 80000 THEN '3: 60k to 80k'
        WHEN annual_inc < 120000 THEN '4: 80k to 120k'
        ELSE '5: 120k and over'
    END AS income_band,
    COUNT(*) AS loans,
    ROUND(AVG(is_default) * 100, 1) AS default_pct,
    ROUND((SUM(total_pymnt) - SUM(loan_amnt)) / SUM(loan_amnt) * 100, 1) AS net_return_pct
FROM loans
WHERE annual_inc IS NOT NULL
GROUP BY income_band
ORDER BY income_band
""")

,income_band,loans,default_pct,net_return_pct
0,1: under 40k,210374,23.7,0.6
1,2: 40k to 60k,355532,21.7,1.9
2,3: 60k to 80k,307925,20.1,2.4
3,4: 80k to 120k,301795,17.6,3.5
4,5: 120k and over,169684,15.6,4.2


In [27]:
q("""
SELECT
    addr_state,
    COUNT(*) AS loans,
    ROUND(AVG(is_default) * 100, 1) AS default_pct,
    ROUND((SUM(total_pymnt) - SUM(loan_amnt)) / SUM(loan_amnt) * 100, 1) AS net_return_pct
FROM loans
GROUP BY addr_state
HAVING COUNT(*) > 5000
ORDER BY default_pct DESC
""")

,addr_state,loans,default_pct,net_return_pct
0,MS,6588,26.1,-2.2
1,AR,10047,24.1,1.0
2,AL,16613,23.6,1.5
3,OK,12281,23.5,0.9
4,LA,15499,23.2,0.8
5,NY,109842,22.0,1.6
6,NV,20267,21.9,0.7
7,FL,95606,21.5,1.2
8,TN,20385,21.4,3.3
9,IN,21716,21.4,2.2


In [28]:
import os
os.makedirs("exports", exist_ok=True)

def summary(keys, where="TRUE", min_loans=0):
    select_part = ", ".join(f"{expr} AS {name}" for name, expr in keys.items())
    group_part = ", ".join(str(i + 1) for i in range(len(keys)))
    return f"""
        SELECT {select_part},
               COUNT(*) AS loans,
               ROUND(AVG(is_default) * 100, 1) AS default_pct,
               ROUND(AVG(int_rate_pct), 1) AS avg_rate_pct,
               ROUND((SUM(total_pymnt) - SUM(loan_amnt)) / SUM(loan_amnt) * 100, 1) AS net_return_pct
        FROM loans
        WHERE {where}
        GROUP BY {group_part}
        HAVING COUNT(*) >= {min_loans}
        ORDER BY {group_part}
    """

dti_case = """CASE
    WHEN dti < 10 THEN '1: under 10'
    WHEN dti < 20 THEN '2: 10 to 20'
    WHEN dti < 30 THEN '3: 20 to 30'
    WHEN dti < 40 THEN '4: 30 to 40'
    ELSE '5: 40 and over' END"""

income_case = """CASE
    WHEN annual_inc < 40000 THEN '1: under 40k'
    WHEN annual_inc < 60000 THEN '2: 40k to 60k'
    WHEN annual_inc < 80000 THEN '3: 60k to 80k'
    WHEN annual_inc < 120000 THEN '4: 80k to 120k'
    ELSE '5: 120k and over' END"""

tables = {
    "overall":        summary({"all_loans": "'All loans'"}),
    "by_grade":       summary({"grade": "grade"}),
    "by_term_grade":  summary({"term": "term", "grade": "grade"}),
    "by_year_36m":    summary({"issue_year": "EXTRACT(year FROM issue_date)"},
                              where="term = '36 months' AND issue_date >= DATE '2009-01-01' AND issue_date < DATE '2016-01-01'"),
    "by_purpose":     summary({"purpose": "purpose"}, min_loans=5000),
    "by_dti":         summary({"dti_band": dti_case}, where="dti >= 0"),
    "by_income":      summary({"income_band": income_case}, where="annual_inc IS NOT NULL"),
    "by_state":       summary({"addr_state": "addr_state"}, min_loans=5000),
}

for name, sql in tables.items():
    df_out = q(sql)
    df_out.to_csv(f"exports/{name}.csv", index=False)
    print(name, df_out.shape)

overall (1, 5)
by_grade (7, 5)
by_term_grade (14, 6)
by_year_36m (7, 5)
by_purpose (11, 5)
by_dti (5, 5)
by_income (5, 5)
by_state (38, 5)
